# 0.0 Imports

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn import metrics as mt

# 0.1 - Load Datasets

In [4]:
actual_folder = Path.cwd()
print(actual_folder)

main_folder = actual_folder.parent.parent
print(main_folder)

path_folder_dataset = main_folder / 'dataset' / 'regression_datasets'

if path_folder_dataset.exists():
    print(f'Caminho existente: {path_folder_dataset}')
else:
    print(f'Erro: O arquivo não foi encontrado {path_folder_dataset}')

d:\repos\projetos\ml_trials_algorithm\notebooks\regressao
d:\repos\projetos\ml_trials_algorithm
Caminho existente: d:\repos\projetos\ml_trials_algorithm\dataset\regression_datasets


In [5]:
#Carrega os datasets da pasta - Treinamento / Validação e Teste

dataset_traning_X = path_folder_dataset / 'a_traninig' / 'X_training.csv'
dataset_traning_y = path_folder_dataset / 'a_traninig' / 'y_training.csv'

dataset_validation_X = path_folder_dataset / 'b_validation' / 'X_validation.csv'
dataset_validation_y = path_folder_dataset / 'b_validation' / 'y_validation.csv'

dataset_test_X = path_folder_dataset / 'c_test' / 'X_test.csv'
dataset_test_y = path_folder_dataset / 'c_test' / 'y_test.csv'

X_train = pd.read_csv(dataset_traning_X)
y_train = pd.read_csv(dataset_traning_y)

X_val = pd.read_csv(dataset_validation_X)
y_val = pd.read_csv(dataset_validation_y)

X_test = pd.read_csv(dataset_test_X)
y_test = pd.read_csv(dataset_test_y)

# 1.0 - Training Algorithm

## Passo 1 — Treino com parâmetros default

In [6]:
# Instanciar o modelo com parâmetros default
model_def = RandomForestRegressor(random_state=42)

# Treinar com dados de treino
model_def.fit(X_train, y_train.values.ravel())

# Predições no treino e na validação
yhat_train_def = model_def.predict(X_train)
yhat_val_def   = model_def.predict(X_val)

## Passo 2 — Performance no treino (default)

In [7]:
# Métricas no conjunto de TREINO com parâmetros default
r2_train_def   = mt.r2_score(y_train, yhat_train_def)
mse_train_def  = mt.mean_squared_error(y_train, yhat_train_def)
rmse_train_def = np.sqrt(mse_train_def)
mae_train_def  = mt.mean_absolute_error(y_train, yhat_train_def)
mape_train_def = mt.mean_absolute_percentage_error(y_train, yhat_train_def)

print("--- Performance no Treino (Default) ---")
print(f"R²:   {r2_train_def:.4f}")
print(f"MSE:  {mse_train_def:.2f}")
print(f"RMSE: {rmse_train_def:.2f}")
print(f"MAE:  {mae_train_def:.2f}")
print(f"MAPE: {mape_train_def * 100:.2f}%")

--- Performance no Treino (Default) ---
R²:   0.9029
MSE:  46.41
RMSE: 6.81
MAE:  4.87
MAPE: 254.88%


## Passo 3 — Performance na validação (default)

In [8]:
# Métricas no conjunto de VALIDAÇÃO com parâmetros default
r2_val_def   = mt.r2_score(y_val, yhat_val_def)
mse_val_def  = mt.mean_squared_error(y_val, yhat_val_def)
rmse_val_def = np.sqrt(mse_val_def)
mae_val_def  = mt.mean_absolute_error(y_val, yhat_val_def)
mape_val_def = mt.mean_absolute_percentage_error(y_val, yhat_val_def)

print("--- Performance na Validação (Default) ---")
print(f"R²:   {r2_val_def:.4f}")
print(f"MSE:  {mse_val_def:.2f}")
print(f"RMSE: {rmse_val_def:.2f}")
print(f"MAE:  {mae_val_def:.2f}")
print(f"MAPE: {mape_val_def * 100:.2f}%")

--- Performance na Validação (Default) ---
R²:   0.3344
MSE:  317.83
RMSE: 17.83
MAE:  13.02
MAPE: 705.75%


## Passo 4 — Ajuste de hiperparâmetros

In [9]:
print("--- Testando Múltiplos Hiperparâmetros na Random Forest (Regressão) ---")
melhor_n_estimators = 100
melhor_max_depth = 5
melhor_min_samples_split = 2
melhor_r2_val = -999

# Listas de hiperparâmetros para o loop triplo
list_n_estimators = [100, 200, 300]
list_max_depth = [3, 5, 8, 10]
list_min_samples_split = [2, 5, 10]

# LOOP TRIPLO: Cruza N_Estimators x Max_Depth x Min_Samples_Split
for n_est in list_n_estimators:
    for md in list_max_depth:
        for mss in list_min_samples_split:
            
            # Instanciando o Regressor de Floresta
            # n_jobs=-1 usa todos os núcleos do processador para acelerar o treino
            model = RandomForestRegressor(
                n_estimators=n_est,
                max_depth=md,
                min_samples_split=mss,
                n_jobs=-1,
                random_state=42
            )
            
            try:
                model.fit(X_train, y_train.values.ravel())
                
                yhat_train = model.predict(X_train)
                yhat_val = model.predict(X_val)
                
                r2_train = mt.r2_score(y_train, yhat_train)
                r2_val = mt.r2_score(y_val, yhat_val)
                rmse_val = np.sqrt(mt.mean_squared_error(y_val, yhat_val))
                
                print(f"N_Est: {n_est} | Max_Depth: {md:2d} | Min_Split: {mss:2d} | R² Treino: {r2_train:.4f} | R² Val: {r2_val:.4f} | RMSE Val: {rmse_val:.2f}")
                
                if r2_val > melhor_r2_val:
                    melhor_r2_val = r2_val
                    melhor_n_estimators = n_est
                    melhor_max_depth = md
                    melhor_min_samples_split = mss
                    
            except Exception as e:
                print(f"Erro na combinação N_Est {n_est} | Max_Depth {md} | Min_Split {mss}: {e}")

print("=" * 85)
print(f"-> VENCEDOR DO ENSAIO RANDOM FOREST:")
print(f"Melhor N_Estimators: {melhor_n_estimators}")
print(f"Melhor Max_Depth: {melhor_max_depth}")
print(f"Melhor Min_Samples_Split: {melhor_min_samples_split}")
print(f"Maior R² obtido na Validação: {melhor_r2_val:.4f}\n")

--- Testando Múltiplos Hiperparâmetros na Random Forest (Regressão) ---
N_Est: 100 | Max_Depth:  3 | Min_Split:  2 | R² Treino: 0.0767 | R² Val: 0.0665 | RMSE Val: 21.11
N_Est: 100 | Max_Depth:  3 | Min_Split:  5 | R² Treino: 0.0767 | R² Val: 0.0665 | RMSE Val: 21.11
N_Est: 100 | Max_Depth:  3 | Min_Split: 10 | R² Treino: 0.0767 | R² Val: 0.0665 | RMSE Val: 21.11
N_Est: 100 | Max_Depth:  5 | Min_Split:  2 | R² Treino: 0.1411 | R² Val: 0.0987 | RMSE Val: 20.75
N_Est: 100 | Max_Depth:  5 | Min_Split:  5 | R² Treino: 0.1411 | R² Val: 0.0987 | RMSE Val: 20.75
N_Est: 100 | Max_Depth:  5 | Min_Split: 10 | R² Treino: 0.1407 | R² Val: 0.0987 | RMSE Val: 20.75
N_Est: 100 | Max_Depth:  8 | Min_Split:  2 | R² Treino: 0.3102 | R² Val: 0.1595 | RMSE Val: 20.03
N_Est: 100 | Max_Depth:  8 | Min_Split:  5 | R² Treino: 0.3081 | R² Val: 0.1587 | RMSE Val: 20.04
N_Est: 100 | Max_Depth:  8 | Min_Split: 10 | R² Treino: 0.3008 | R² Val: 0.1575 | RMSE Val: 20.06
N_Est: 100 | Max_Depth: 10 | Min_Split:  2 | R

## Passo 5 — Performance com modelo tunado (treino e validação)

In [10]:
# Modelo com os melhores hiperparâmetros encontrados, treinado apenas em X_train
model_tunado = RandomForestRegressor(
    n_estimators=melhor_n_estimators,
    max_depth=melhor_max_depth,
    min_samples_split=melhor_min_samples_split,
    n_jobs=-1,
    random_state=42
)
model_tunado.fit(X_train, y_train.values.ravel())

yhat_train_tunado = model_tunado.predict(X_train)
yhat_val_tunado   = model_tunado.predict(X_val)

r2_train_tunado   = mt.r2_score(y_train, yhat_train_tunado)
mse_train_tunado  = mt.mean_squared_error(y_train, yhat_train_tunado)
rmse_train_tunado = np.sqrt(mse_train_tunado)
mae_train_tunado  = mt.mean_absolute_error(y_train, yhat_train_tunado)
mape_train_tunado = mt.mean_absolute_percentage_error(y_train, yhat_train_tunado)

r2_val_tunado   = mt.r2_score(y_val, yhat_val_tunado)
mse_val_tunado  = mt.mean_squared_error(y_val, yhat_val_tunado)
rmse_val_tunado = np.sqrt(mse_val_tunado)
mae_val_tunado  = mt.mean_absolute_error(y_val, yhat_val_tunado)
mape_val_tunado = mt.mean_absolute_percentage_error(y_val, yhat_val_tunado)

print("--- Performance com Modelo Tunado ---")
print(f"Treino    → R²: {r2_train_tunado:.4f} | RMSE: {rmse_train_tunado:.2f} | MAE: {mae_train_tunado:.2f} | MAPE: {mape_train_tunado*100:.2f}%")
print(f"Validação → R²: {r2_val_tunado:.4f} | RMSE: {rmse_val_tunado:.2f} | MAE: {mae_val_tunado:.2f} | MAPE: {mape_val_tunado*100:.2f}%")

--- Performance com Modelo Tunado ---
Treino    → R²: 0.4681 | RMSE: 15.94 | MAE: 12.68 | MAPE: 572.57%
Validação → R²: 0.2133 | RMSE: 19.38 | MAE: 15.28 | MAPE: 787.52%


## Passo 6 — União treino + validação

In [11]:
# Unir treino + validação para formar o conjunto final de treinamento
X_train_final = pd.concat([X_train, X_val])
y_train_final = pd.concat([y_train, y_val])

print(f"X_train_final shape: {X_train_final.shape}")
print(f"y_train_final shape: {y_train_final.shape}")

X_train_final shape: (15068, 13)
y_train_final shape: (15068, 1)


## Passo 7 — Retreino com melhores parâmetros

In [12]:
# Retreino com os melhores hiperparâmetros sobre o conjunto final (treino + validação)
model_final = RandomForestRegressor(
    n_estimators=melhor_n_estimators,
    max_depth=melhor_max_depth,
    min_samples_split=melhor_min_samples_split,
    n_jobs=-1,
    random_state=42
)
model_final.fit(X_train_final, y_train_final.values.ravel())

,n_estimators,300
,criterion,'squared_error'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


## Passo 8 — Performance no teste

In [13]:
# Predição no conjunto de teste (única vez que X_test é tocado)
yhat_test = model_final.predict(X_test)

# Métricas no conjunto de TESTE
r2_test   = mt.r2_score(y_test, yhat_test)
mse_test  = mt.mean_squared_error(y_test, yhat_test)
rmse_test = np.sqrt(mse_test)
mae_test  = mt.mean_absolute_error(y_test, yhat_test)
mape_test = mt.mean_absolute_percentage_error(y_test, yhat_test)

print("--- Performance no Teste (Final) ---")
print(f"R²:   {r2_test:.4f}")
print(f"MSE:  {mse_test:.2f}")
print(f"RMSE: {rmse_test:.2f}")
print(f"MAE:  {mae_test:.2f}")
print(f"MAPE: {mape_test * 100:.2f}%")

--- Performance no Teste (Final) ---
R²:   0.2462
MSE:  367.05
RMSE: 19.16
MAE:  15.26
MAPE: 715.39%


## Passo 9 — Quadro Comparativo — Diagnóstico Completo

In [14]:
data_comparativo = {
    'Conjunto': ['Treino (Default)', 'Validação (Default)', 'Treino (Tunado)', 'Validação (Tunado)', 'Teste (Final)'],
    'R²':    [r2_train_def,   r2_val_def,   r2_train_tunado,   r2_val_tunado,   r2_test],
    'RMSE':  [rmse_train_def, rmse_val_def, rmse_train_tunado, rmse_val_tunado, rmse_test],
    'MAE':   [mae_train_def,  mae_val_def,  mae_train_tunado,  mae_val_tunado,  mae_test],
    'MAPE':  [f'{mape_train_def*100:.2f}%', f'{mape_val_def*100:.2f}%',
              f'{mape_train_tunado*100:.2f}%', f'{mape_val_tunado*100:.2f}%',
              f'{mape_test*100:.2f}%'],
}
df_comparativo = pd.DataFrame(data_comparativo)
print("\n--- Quadro Comparativo — Diagnóstico Completo ---")
print(df_comparativo.to_string(index=False))


--- Quadro Comparativo — Diagnóstico Completo ---
           Conjunto       R²      RMSE       MAE    MAPE
   Treino (Default) 0.902913  6.812413  4.866513 254.88%
Validação (Default) 0.334400 17.827835 13.015607 705.75%
    Treino (Tunado) 0.468131 15.944901 12.680885 572.57%
 Validação (Tunado) 0.213330 19.381533 15.283240 787.52%
      Teste (Final) 0.246152 19.158528 15.264181 715.39%
